<a href="https://colab.research.google.com/github/majavier26/DSProjects/blob/main/Weather%20dashboard/Updating_weather_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
! pip install openmeteo-requests
! pip install requests-cache retry-requests

  Using cached openmeteo_requests-1.7.4-py3-none-any.whl.metadata (11 kB)
  Using cached niquests-3.15.2-py3-none-any.whl.metadata (16 kB)
  Using cached openmeteo_sdk-1.23.0-py3-none-any.whl.metadata (935 bytes)
  Using cached urllib3_future-2.14.908-py3-none-any.whl.metadata (16 kB)
  Using cached wassima-2.0.2-py3-none-any.whl.metadata (3.7 kB)
  Using cached jh2-5.0.10-cp37-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (4.0 kB)
  Using cached qh3-1.5.6-cp37-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (4.8 kB)
Using cached openmeteo_requests-1.7.4-py3-none-any.whl (7.0 kB)
Using cached niquests-3.15.2-py3-none-any.whl (167 kB)
Using cached openmeteo_sdk-1.23.0-py3-none-any.whl (18 kB)
Using cached urllib3_future-2.14.908-py3-none-any.whl (683 kB)
Using cached wassima-2.0.2-py3-none-any.whl (145 kB)
Using cached jh2-5.0.10-cp37-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (394 kB)
Using cached qh3-1.5.6-cp37-abi3-manylinux_2_17_x86_64.manylinux2

In [3]:
# Packages
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import numpy as np
import math
import seaborn as sns
import matplotlib.colors as cm
import calendar
import time
import os
import zipfile
from natsort import natsorted
import matplotlib.ticker as mticker
import matplotlib.gridspec as gridspec
import csv
import datetime
import matplotlib.dates as mdates
import itertools
from datetime import date, timedelta

# OpenMeteo API
import openmeteo_requests
import requests_cache
from retry_requests import retry

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Philippine Weather Map Updater

## Initializing the map

In [5]:
# Initializing the map
filepath = r'/content/drive/MyDrive/Philippines shapefile/gadm41_PHL_1.shp'
PHL = gpd.read_file(filepath)

# Meshgrid initialization
dx_theo = 0.25
y_values = np.arange(4, 22, dx_theo)
x_values = np.arange(116, 127, dx_theo)

# Unraveling the meshgrid to get each point
x_mesh, y_mesh = np.meshgrid(x_values, y_values)
xy_points = np.vstack([x_mesh.ravel(), y_mesh.ravel()]).T

# Using the actual spacing for the map
dx, dy = np.abs(x_values[1] - x_values[0]), np.abs(y_values[1] - y_values[0])

# Converting the mesh of points into a geodataframe
geo_points = gpd.GeoDataFrame(xy_points, geometry=gpd.points_from_xy(xy_points[:, 0], xy_points[:, 1]))
geo_points.crs = PHL.crs # making the coordinate system of the two the same

# Intersecting the points to the shapefile
phl_points = gpd.sjoin(geo_points, PHL, how='inner', predicate='within')

# Getting only the coordinates and not the other columns in the PHL shapefile
points_in_phl = phl_points[[0, 1, 'geometry']]
points_in_phl.columns = ['coord.lon', 'coord.lat', 'geometry']
points_in_phl.reset_index(drop=True, inplace=True)

# Making the points have a buffer
square_points = points_in_phl.to_crs(crs=PHL.crs).buffer(dx_theo/2, cap_style=3) # dx_theo/2 is the size of buffer, cap_style=3 makes it square
square_points.reset_index(drop=True, inplace=True)

/tmp/ipython-input-4170175810.py:30: UserWarning: Geometry is in a geographic CRS. Results from 'buffer' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  square_points = points_in_phl.to_crs(crs=PHL.crs).buffer(dx_theo/2, cap_style=3) # dx_theo/2 is the size of buffer, cap_style=3 makes it square


The latitudes ang longitudes of the points are saved in:

In [6]:
points_in_phl

,coord.lon,coord.lat,geometry
0,119.50,4.75,POINT (119.5 4.75)
1,120.00,5.00,POINT (120 5)
2,125.25,5.75,POINT (125.25 5.75)
3,125.50,5.75,POINT (125.5 5.75)
4,121.00,6.00,POINT (121 6)
...,...,...,...
391,122.00,18.25,POINT (122 18.25)
392,122.25,18.25,POINT (122.25 18.25)
393,120.75,18.50,POINT (120.75 18.5)
394,121.00,18.50,POINT (121 18.5)


## Getting and updating data

In [7]:
# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

def get_data(coord_lat, coord_lon, date_start, date_end):
  # Make sure all required weather variables are listed here
  # The order of variables in hourly or daily is important to assign them correctly below
  # Coordinates are floats
  # Date is of the format YYYY-MM-DD and should be a string
  url = "https://historical-forecast-api.open-meteo.com/v1/forecast"
  params = {
    "latitude": coord_lat, # x-values
    "longitude": coord_lon, # y-values
    "start_date": date_start,
    "end_date": date_end,
    "daily": ["temperature_2m_max", "temperature_2m_min", "rain_sum", "wind_speed_10m_max"],
    "timezone": "Asia/Singapore"
  }
  responses = openmeteo.weather_api(url, params=params)

  # Process first location. Add a for-loop for multiple locations or weather models
  response = responses[0]
  print(f"Coordinates {response.Latitude()}°N {response.Longitude()}°E")
  print(f"Elevation {response.Elevation()} m asl")
  print(f"Timezone {response.Timezone()} {response.TimezoneAbbreviation()}")
  print(f"Timezone difference to GMT+0 {response.UtcOffsetSeconds()} s")

  # Process daily data. The order of variables needs to be the same as requested.
  daily = response.Daily()
  daily_temperature_2m_max = daily.Variables(0).ValuesAsNumpy()
  daily_temperature_2m_min = daily.Variables(1).ValuesAsNumpy()
  daily_rain_sum = daily.Variables(2).ValuesAsNumpy()
  daily_wind_speed_10m_max = daily.Variables(3).ValuesAsNumpy()

  daily_data = {"date": pd.date_range(
    start = pd.to_datetime(daily.Time(), unit = "s", utc = True),
    end = pd.to_datetime(daily.TimeEnd(), unit = "s", utc = True),
    freq = pd.Timedelta(seconds = daily.Interval()),
    inclusive = "left"
  )}
  daily_data["temperature_2m_max"] = daily_temperature_2m_max
  daily_data["temperature_2m_min"] = daily_temperature_2m_min
  daily_data["rain_sum"] = daily_rain_sum
  daily_data["wind_speed_10m_max"] = daily_wind_speed_10m_max

  daily_dataframe = pd.DataFrame(data = daily_data)
  return daily_dataframe

In [8]:
def get_weather_data(point_list, year):
  iteration = 0
  all_weather_data = []

  # Start and end dates
  if year == date.today().year: # if current year
    date_start = f'2024-01-01'
    date_end = date.today().strftime("%Y-%m-%d") # date today
  else:
    date_start = f'{year}-01-01'
    date_end = f'{year}-12-30'

  for index in range(len(point_list)):
    # Loop sleeps for 1 minute every 50 cities for the API to recover
    if iteration % 60 == 0 and iteration !=0:
      print(f'Waiting...')
      time.sleep(60)
      print(f'Iteration: {iteration}')
      city_weather = get_data(point_list.iloc[index]['coord.lat'], point_list.iloc[index]['coord.lon'], date_start, date_end)
      all_weather_data.append(city_weather)
      iteration += 1
    elif iteration == len(point_list):
      print(f'Waiting...')
      time.sleep(60)
    else:
      print(f'Iteration: {iteration}')
      city_weather = get_data(point_list.iloc[index]['coord.lat'], point_list.iloc[index]['coord.lon'], date_start, date_end)
      all_weather_data.append(city_weather)
      iteration += 1

  return all_weather_data

We then need to get the folders in our database:

In [9]:
openmeteo_folders = os.listdir('/content/drive/MyDrive/Colab Notebooks/Weather clustering/OpenMeteo Data')
openmeteo_folders.sort()
openmeteo_folders

['data_2021', 'data_2022', 'data_2023', 'data_2024']

The function `read_present_data` reads the csv files of the present data and drops the duplicated rows according to its `date`.

In [10]:
def read_present_data():
  data_2024 = []

  for iter in range(len(points_in_phl)):
      csv_name = f'iteration {iter}.csv'
      # Read the CSV, potentially making the first column the index
      data = pd.read_csv(f'/content/drive/MyDrive/Colab Notebooks/Weather clustering/OpenMeteo Data/data_2024/{csv_name}') # Removed index_col=0

      # Drop duplicates
      data = data.drop_duplicates(subset='date', keep="last")

      # If 'Unnamed: 0' or 'level_0' exists (from previous saves), drop it before resetting
      # This handles cases where reset_index created this column
      if 'Unnamed: 0' in data.columns:
          data = data.drop('Unnamed: 0', axis=1)
      if 'level_0' in data.columns:
          data = data.drop('level_0', axis=1)
      if 'index' in data.columns: # Also check for 'index' which is another default name
          data = data.drop('index', axis=1)

      # Reset the index, ensuring the new index is named 'index' or similar and doesn't conflict
      data = data.reset_index(drop=True) # drop=True prevents adding the old index as a column

      # Save data with dropped duplicates and clean index
      data.to_csv(f'/content/drive/MyDrive/Colab Notebooks/Weather clustering/OpenMeteo Data/data_2024/{csv_name}', index=False) # index=False to avoid writing the index as a column

      print(f'Successfully read iteration {iter}.csv and dropped its duplicates.')

      # Append data in data_2024 list
      data_2024.append(data)

  return data_2024

The folder for present data will be saved in the list `data_2024`.

In [11]:
data_2024 = read_present_data()

Successfully read iteration 0.csv and dropped its duplicates.
Successfully read iteration 1.csv and dropped its duplicates.
Successfully read iteration 2.csv and dropped its duplicates.
Successfully read iteration 3.csv and dropped its duplicates.
Successfully read iteration 4.csv and dropped its duplicates.
Successfully read iteration 5.csv and dropped its duplicates.
Successfully read iteration 6.csv and dropped its duplicates.
Successfully read iteration 7.csv and dropped its duplicates.
Successfully read iteration 8.csv and dropped its duplicates.
Successfully read iteration 9.csv and dropped its duplicates.
Successfully read iteration 10.csv and dropped its duplicates.
Successfully read iteration 11.csv and dropped its duplicates.
Successfully read iteration 12.csv and dropped its duplicates.
Successfully read iteration 13.csv and dropped its duplicates.
Successfully read iteration 14.csv and dropped its duplicates.
Successfully read iteration 15.csv and dropped its duplicates.
Su

What is the latest date in our csv files?

In [13]:
np.unique([pd.to_datetime(data['date']).dt.tz_localize(None).iloc[-1] for data in data_2024])

array([Timestamp('2025-05-22 16:00:00')], dtype=object)

In [29]:
def get_missing_data(point_list, date_start, date_end):
      iteration = 0
      all_weather_data = []
      date_difference = (pd.to_datetime(date_end) - pd.to_datetime(date_start)).days
      for index in range(len(point_list)):
        # Determines if date_difference is too big so for loop doesn't sleep as much
        if date_difference < 20:
          rest_number = 500
        elif date_difference >= 20 and date_difference < 100:
          rest_number = 250
        elif date_difference >= 100 and date_difference < 200:
          rest_number = 100
        else:
          rest_number = 50
        # Loop sleeps for 1 minute every n cities for the API to recover
        if iteration % rest_number == 0 and iteration !=0:
          print(f'Waiting...')
          time.sleep(60)
          print(f'Iteration: {iteration}')
          city_weather = get_data(point_list.iloc[index]['coord.lat'], point_list.iloc[index]['coord.lon'], date_start, date_end)
          all_weather_data.append(city_weather)
          iteration += 1
        elif iteration == len(point_list):
          print(f'Waiting...')
          time.sleep(60)
        else:
          print(f'Iteration: {iteration}')
          city_weather = get_data(point_list.iloc[index]['coord.lat'], point_list.iloc[index]['coord.lon'], date_start, date_end)
          all_weather_data.append(city_weather)
          iteration += 1

      return all_weather_data

In [ ]:
# Function for updating data
def update_data():
  date_today = date.today()
  date_yesterday = date_today - timedelta(days=2)
  date_yesterday_str = date_yesterday.strftime('%Y-%m-%d')

  date_current = pd.to_datetime(data_2024[0]['date']).dt.tz_localize(None).iloc[-1] # most current data in data_2024
  print(date_current)
  date_current_str = date_current.strftime('%Y-%m-%d')
  date_next = date_current + timedelta(days=1)
  date_next_str = date_next.strftime('%Y-%m-%d')

  if date_yesterday_str == date_current_str:
    print('Data is already updated!')
  else:
    print('Updating data...')
    # New data
    missing_data = get_missing_data(points_in_phl, date_next_str, date_yesterday_str)
    print('Done gathering data!')
    # Update data
    for iter in range(len(points_in_phl)):
        csv_name = f'iteration {iter}.csv'
        file_path = f'/content/drive/MyDrive/Colab Notebooks/Weather clustering/OpenMeteo Data/data_2024/{csv_name}'
        # Read existing data
        existing_df = pd.read_csv(file_path)
        # Get new data
        new_df = missing_data[iter]
        # Combine and remove duplicates
        combined_df = pd.concat([existing_df, new_df], ignore_index=True)
        combined_df = combined_df.drop_duplicates(subset=['date'], keep='first')
        # Save without index
        combined_df.to_csv(file_path, index=False)
        print(f'Updated {csv_name}')
    print('Done updating data to present!')

In [ ]:
# Updating data
# update_data()

In [27]:
# Reading updated data
data_2024 = read_present_data()

Successfully read iteration 0.csv and dropped its duplicates.
Successfully read iteration 1.csv and dropped its duplicates.
Successfully read iteration 2.csv and dropped its duplicates.
Successfully read iteration 3.csv and dropped its duplicates.
Successfully read iteration 4.csv and dropped its duplicates.
Successfully read iteration 5.csv and dropped its duplicates.
Successfully read iteration 6.csv and dropped its duplicates.
Successfully read iteration 7.csv and dropped its duplicates.
Successfully read iteration 8.csv and dropped its duplicates.
Successfully read iteration 9.csv and dropped its duplicates.
Successfully read iteration 10.csv and dropped its duplicates.
Successfully read iteration 11.csv and dropped its duplicates.
Successfully read iteration 12.csv and dropped its duplicates.
Successfully read iteration 13.csv and dropped its duplicates.
Successfully read iteration 14.csv and dropped its duplicates.
Successfully read iteration 15.csv and dropped its duplicates.
Su

### Data Fixer



Originally, the data did not have an index column, only these columns where present:
- date
- temperature_2m_max
- temperature_2m_min
- rain_sum
- wind_speed_10m_max

However, a bug in my `update_data` function appended dataframes with the index column so now the second part of the csv files have these:
- index
- temperature_2m_max
- temperature_2m_min
- rain_sum
- wind_speed_10m_max

This attempts to fix it by going over line by line and removing the index.

In [25]:
def remove_index_from_line(line_list):
  """
  Removes the index column from a list of strings.

  Args:
    line_list (str[]): List of strings representing a line in a CSV file.

  Returns:
    str[]: List of strings with the index column removed.
  """
  # Keep the header
  header = lines[0]

  # Process data rows
  clean_rows = [header]

  for line in lines[1:]:  # Skip header
      parts = line.strip().split(',')

      # Check if first column is a date (contains '-' or ':') or an index (just a number)
      first_col = parts[0]

      # If it's just digits (an index), remove it
      if first_col.isdigit():
          # Remove first column (the index)
          clean_line = ','.join(parts[1:]) + '\n'
          clean_rows.append(clean_line)
      else:
          # Keep the line as is (first column is already the date)
          clean_rows.append(line)

  return clean_rows

In [26]:
# Fix all iteration files
for iter in range(len(points_in_phl)):
    csv_name = f'iteration {iter}.csv'
    file_path = f'/content/drive/MyDrive/Colab Notebooks/Weather clustering/OpenMeteo Data/data_2024/{csv_name}'

    # Read the entire file as text lines
    with open(file_path, 'r') as f:
        lines = f.readlines()

    # Clean the data
    clean_rows = remove_index_from_line(lines)

    # Write back the cleaned data
    with open(file_path, 'w') as f:
        f.writelines(clean_rows)

    # Now read it properly and remove duplicates
    df = pd.read_csv(file_path)
    df = df.drop_duplicates(subset=['date'], keep='first')
    df.to_csv(file_path, index=False)

    print(f'Fixed {csv_name}')

Fixed iteration 0.csv
Fixed iteration 1.csv
Fixed iteration 2.csv
Fixed iteration 3.csv
Fixed iteration 4.csv
Fixed iteration 5.csv
Fixed iteration 6.csv
Fixed iteration 7.csv
Fixed iteration 8.csv
Fixed iteration 9.csv
Fixed iteration 10.csv
Fixed iteration 11.csv
Fixed iteration 12.csv
Fixed iteration 13.csv
Fixed iteration 14.csv
Fixed iteration 15.csv
Fixed iteration 16.csv
Fixed iteration 17.csv
Fixed iteration 18.csv
Fixed iteration 19.csv
Fixed iteration 20.csv
Fixed iteration 21.csv
Fixed iteration 22.csv
Fixed iteration 23.csv
Fixed iteration 24.csv
Fixed iteration 25.csv
Fixed iteration 26.csv
Fixed iteration 27.csv
Fixed iteration 28.csv
Fixed iteration 29.csv
Fixed iteration 30.csv
Fixed iteration 31.csv
Fixed iteration 32.csv
Fixed iteration 33.csv
Fixed iteration 34.csv
Fixed iteration 35.csv
Fixed iteration 36.csv
Fixed iteration 37.csv
Fixed iteration 38.csv
Fixed iteration 39.csv
Fixed iteration 40.csv
Fixed iteration 41.csv
Fixed iteration 42.csv
Fixed iteration 43.cs